# Experiment Results Analysis 

The object of this notebook is to conduct the final experiment analysis in order to test hypothesis regarding algorithm performance.

In this notebook the different executed batches of experiments will be consolidated in a unique database to analyze its results.


## Analysis Conducted

1. Compare AUC for each algorithm in each enviroment (Comparisons are conducted across algorithms in the same environment)

2. Compare the Mean Best So Far Curve in each enviroment (Comparisons are conducted across algorithms in the same environment) with confidence intervals

3. Perform the wilcoxon Rank Sum test in each enviroment for selected pairwise comaprisons (Comparisons are conducted across algorithms in the same environment) with confidence intervals


## Algorithms to compare

1. Binary Particle Swarm Optimization - (PSO)

2. Binary Global Random Search - (Random)

3. Standard Binary Representation GA (Generic)

4. Standard Mixed Integer Representation GA (mixed_generic)

5. Macro_Micro CX Operator GA - (macro_micro)

6. Macro CX Operator GA - Deactivates Micro CX operator - (micro)

7. Micro CX Operator GA - Deactivates Macro CX operator - (macro)

8. Parents CX Operator - Uses a parent recombination CX Operator (recomb)

## 1. Library Imports

In [1]:
import pandas as pd 
import numpy as np
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt
import seaborn as sns
import os 
import ast
from typing import List, Dict 
import re
import zipfile

## 2. Data Import

In [2]:
# Check wd
print(os.getcwd())



c:\Users\andre\Repositories\FTZ_model_2.0\final_results\experiment


In [3]:


# Route to Folder 
base_path = "data_exp/hyperdense"

# Verify 
print("Working dir:", os.getcwd())
print("Looking for zips inside:", base_path)

# Zip List 
zip_files = [
    os.path.join(base_path, f)
    for f in os.listdir(base_path)
    if f.lower().endswith(".zip")
]

print("\nFound ZIPs:")
for z in zip_files:
    print(" -", z)

all_dfs = []

for zip_path in zip_files:
    print(f"\n🟦 Processing ZIP: {zip_path}")

    with zipfile.ZipFile(zip_path, "r") as z:
        # Buscar el results.csv sin importar la subcarpeta
        results_candidates = [
            name for name in z.namelist()
            if name.lower().endswith("results.csv")
        ]

        if not results_candidates:
            print("⚠ No results.csv found in this ZIP, skipping.")
            continue

        results_path = results_candidates[0]
        print(f"   → results.csv found at: {results_path}")

        # Leer CSV dentro del ZIP
        with z.open(results_path) as f:
            df = pd.read_csv(f)

        # Metadata útil
        df["source_zip"] = os.path.basename(zip_path)
        df["internal_path"] = results_path

        all_dfs.append(df)

# Concatenar todos
df_final = pd.concat(all_dfs, ignore_index=True)

print("Final shape:", df_final.shape)


Working dir: c:\Users\andre\Repositories\FTZ_model_2.0\final_results\experiment
Looking for zips inside: data_exp/hyperdense

Found ZIPs:
 - data_exp/hyperdense\experiment_hyperdense_mixed_generic (1).zip
 - data_exp/hyperdense\final_exp_hyperdense_macromicro.zip
 - data_exp/hyperdense\Hyperdense_pso_generic (1).zip

🟦 Processing ZIP: data_exp/hyperdense\experiment_hyperdense_mixed_generic (1).zip
   → results.csv found at: experiment_hyperdense_mixed_generic/results.csv

🟦 Processing ZIP: data_exp/hyperdense\final_exp_hyperdense_macromicro.zip
   → results.csv found at: final_exp_hyperdense_macromicro/experiment_hyperdensyperdense_macromicro_20251110_203603/results.csv

🟦 Processing ZIP: data_exp/hyperdense\Hyperdense_pso_generic (1).zip
   → results.csv found at: Hyperdense_pso_generic/results.csv
Final shape: (480, 12)


In [4]:
df_all = df_final.copy()

## 3. Clean Data 

Due to a code / paralleization mistake, some rows are duplicated

In [5]:
df_all.head()

,env_name,run,algorithm,seed,best_curve,best_value,best_genome,all_best_genomes,elapsed_sec,success,source_zip,internal_path
0,Hyper-Dense_standard,19,mixed_generic,905430,"[16613.0, 16613.0, 16614.8, 16617.5, 16624.25,...",16687.25,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...",3562.984575,True,experiment_hyperdense_mixed_generic (1).zip,experiment_hyperdense_mixed_generic/results.csv
1,Hyper-Dense_standard,15,mixed_generic,1379495,"[16606.25, 16613.45, 16613.45, 16613.45, 16620...",16687.25,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...",3636.175684,True,experiment_hyperdense_mixed_generic (1).zip,experiment_hyperdense_mixed_generic/results.csv
2,Hyper-Dense_standard,1,mixed_generic,498605,"[16608.05, 16626.5, 16626.95, 16631.9, 16635.5...",16686.80,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...",3692.204867,True,experiment_hyperdense_mixed_generic (1).zip,experiment_hyperdense_mixed_generic/results.csv
3,Hyper-Dense_standard,9,mixed_generic,798166,"[16608.05, 16616.6, 16616.6, 16616.6, 16618.85...",16686.80,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...",3724.974014,True,experiment_hyperdense_mixed_generic (1).zip,experiment_hyperdense_mixed_generic/results.csv
4,Hyper-Dense_standard,12,mixed_generic,1084932,"[16617.95, 16623.8, 16625.15, 16625.15, 16626....",16687.70,"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,...",3753.518041,True,experiment_hyperdense_mixed_generic (1).zip,experiment_hyperdense_mixed_generic/results.csv


In [6]:
df_all.columns

Index(['env_name', 'run', 'algorithm', 'seed', 'best_curve', 'best_value',
       'best_genome', 'all_best_genomes', 'elapsed_sec', 'success',
       'source_zip', 'internal_path'],
      dtype='object')

In [7]:
# Keep the row with the smallest elapsed_sec per (env_name, algorithm, seed, run)
before = len(df_all)
print(f"Total rows before removing duplicates: {before}")

df_tmp = df_all.copy()
df_tmp["elapsed_sec"] = pd.to_numeric(df_tmp.get("elapsed_sec", np.nan), errors="coerce")
df_tmp["_elapsed_sort"] = df_tmp["elapsed_sec"].fillna(np.inf)

df_analysis = (
    df_tmp
    .sort_values(["env_name", "algorithm", "seed", "run", "_elapsed_sort"],
                 ascending=[True, True, True, True, True])
    .drop_duplicates(subset=["env_name", "algorithm", "seed", "run"], keep="first")
    .drop(columns=["_elapsed_sort"])
    .reset_index(drop=True)
)

after = len(df_analysis)
print(f"Removed {before - after} duplicate rows")


Total rows before removing duplicates: 480
Removed 160 duplicate rows


In [8]:
df_analysis.shape

(320, 12)

In [9]:
df_analysis.head()

,env_name,run,algorithm,seed,best_curve,best_value,best_genome,all_best_genomes,elapsed_sec,success,source_zip,internal_path
0,Hyper-Dense_perturbed,2,generic,455222,"[16567.25, 16567.25, 16577.0, 16593.35, 16640....",17235.35,"[1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, ...",[],4180.301175,True,Hyperdense_pso_generic (1).zip,Hyperdense_pso_generic/results.csv
1,Hyper-Dense_perturbed,3,generic,515270,"[16550.3, 16598.9, 16646.0, 16658.6, 16691.15,...",17214.20,"[1, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, ...",[],4347.966424,True,Hyperdense_pso_generic (1).zip,Hyperdense_pso_generic/results.csv
2,Hyper-Dense_perturbed,5,generic,523719,"[16474.7, 16481.15, 16492.55, 16532.15, 16532....",17236.85,"[0, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, ...",[],4289.344795,True,Hyperdense_pso_generic (1).zip,Hyperdense_pso_generic/results.csv
3,Hyper-Dense_perturbed,8,generic,530390,"[16552.25, 16580.6, 16580.6, 16632.95, 16642.5...",17212.10,"[0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, ...",[],3779.478423,True,Hyperdense_pso_generic (1).zip,Hyperdense_pso_generic/results.csv
4,Hyper-Dense_perturbed,21,generic,614958,"[16549.55, 16549.55, 16555.55, 16560.5, 16612....",17216.75,"[1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, ...",[],4349.525722,True,Hyperdense_pso_generic (1).zip,Hyperdense_pso_generic/results.csv


## 4. Graphics of Mean Best So Far Curves

In [10]:
df_graphics = df_analysis[["env_name", "run", "seed", "algorithm", "best_curve", "best_value"]].copy()


In [11]:
df_graphics.head()

,env_name,run,seed,algorithm,best_curve,best_value
0,Hyper-Dense_perturbed,2,455222,generic,"[16567.25, 16567.25, 16577.0, 16593.35, 16640....",17235.35
1,Hyper-Dense_perturbed,3,515270,generic,"[16550.3, 16598.9, 16646.0, 16658.6, 16691.15,...",17214.20
2,Hyper-Dense_perturbed,5,523719,generic,"[16474.7, 16481.15, 16492.55, 16532.15, 16532....",17236.85
3,Hyper-Dense_perturbed,8,530390,generic,"[16552.25, 16580.6, 16580.6, 16632.95, 16642.5...",17212.10
4,Hyper-Dense_perturbed,21,614958,generic,"[16549.55, 16549.55, 16555.55, 16560.5, 16612....",17216.75


In [12]:
df_graphics["best_curve"].dtype


dtype('O')

In [13]:
def safe_parse_with_inf(x):
    if not isinstance(x, str):
        return np.array(x, dtype=float) if isinstance(x, (list, np.ndarray)) else np.array([])

    # Replace 'inf', '-inf', 'nan' with strings that eval can interpret
    x = re.sub(r'\b-inf\b', 'float("-inf")', x)
    x = re.sub(r'\binf\b', 'float("inf")', x)
    x = re.sub(r'\bnan\b', 'float("nan")', x)

    try:
        parsed = ast.literal_eval(x)
        return np.array(parsed, dtype=float)
    except Exception:
        return np.array([])

df_graphics["best_curve"] = df_graphics["best_curve"].apply(safe_parse_with_inf)

In [14]:
df_graphics.columns

Index(['env_name', 'run', 'seed', 'algorithm', 'best_curve', 'best_value'], dtype='object')

### Graphics 

In [15]:
def clean_curve(arr):
    arr = np.array(arr, dtype=float)
    mask_invalid = ~np.isfinite(arr)  # detects inf, -inf, nan
    n_invalid = np.sum(mask_invalid)
    arr = arr[np.isfinite(arr)]       # keep only finite values
    return arr, n_invalid

# Clean curves and count removals
invalid_counts = []
cleaned_curves = []

for idx, arr in enumerate(df_graphics["best_curve"]):
    cleaned, n_invalid = clean_curve(arr)
    invalid_counts.append(n_invalid)
    cleaned_curves.append(cleaned)

df_graphics["best_curve"] = cleaned_curves
df_graphics["invalid_points"] = invalid_counts

print(f"Total invalid values removed: {sum(invalid_counts)}")

Total invalid values removed: 0


In [16]:
from scipy.stats import t

from scipy.stats import t
import numpy as np

def mean_curve_with_ci(curves, debug=False):
    """
    Computes the mean curve and 95% confidence interval across runs.
    Filters out invalid, empty, or malformed curves.
    """

    clean_curves = []
    for i, c in enumerate(curves):
        # Skip non-list/array items or empty ones
        if not isinstance(c, (list, np.ndarray)):
            if debug:
                print(f"⚠️ Run {i} skipped (not list/array): {type(c)} -> {c}")
            continue
        c = np.array(c, dtype=float).flatten()

        # Skip empty or NaN-only curves
        if len(c) == 0 or not np.isfinite(np.sum(c)):
            if debug:
                print(f"⚠️ Run {i} skipped (empty or invalid values): {c}")
            continue

        clean_curves.append(c)

    # If none valid, return empty arrays
    if len(clean_curves) == 0:
        if debug:
            print("❌ No valid curves in group.")
        return np.array([]), np.array([]), np.array([])

    # Truncate to the shortest length to align
    min_len = min(len(c) for c in clean_curves)
    if debug:
        print(f"✅ Using {len(clean_curves)} valid curves, truncated to length {min_len}")

    data = np.stack([c[:min_len] for c in clean_curves], axis=0)

    # Compute statistics
    mean = np.nanmean(data, axis=0)
    std = np.nanstd(data, axis=0, ddof=1)
    n = data.shape[0]
    alpha = 0.05
    tcrit = t.ppf(1 - alpha/2, df=n - 1) if n > 1 else 0
    se = std / np.sqrt(n)
    ci = tcrit * se

    return mean, mean - ci, mean + ci


In [17]:
import matplotlib.pyplot as plt
import numpy as np
import os

# === Output folder on your Desktop ===
desktop = os.path.join(os.path.expanduser("~"), "Desktop")
output_dir = os.path.join(desktop, "FTZ_Plots")
os.makedirs(output_dir, exist_ok=True)
print(f" Plots will be saved in: {output_dir}\n")

# === Group by environment and algorithm ===
grouped = df_graphics.groupby(["env_name", "algorithm"])

# Unique environments
envs = df_graphics["env_name"].unique()

#  Print how many and which environments will be plotted
print(f" Plotting {len(envs)} environments:")
for i, e in enumerate(envs, 1):
    print(f"   {i}. {e}")




 Plots will be saved in: C:\Users\andre\Desktop\FTZ_Plots

 Plotting 2 environments:
   1. Hyper-Dense_perturbed
   2. Hyper-Dense_standard


In [18]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# === Configuración global de estilo ===
plt.style.use("seaborn-v0_8-whitegrid")

plt.rcParams.update({
    "font.size": 14,
    "axes.titlesize": 16,
    "axes.labelsize": 14,
    "legend.fontsize": 12,
    "lines.linewidth": 2,
    "lines.markersize": 6,
    "axes.linewidth": 1.2,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "figure.dpi": 300,
    "savefig.dpi": 400,
    "figure.autolayout": True
})

# === Selección de algoritmos a graficar ===
# Deja la lista vacía [] para graficar todos los algoritmos del DataFrame
algorithms_to_plot = []  # ejemplo
# algorithms_to_plot = []  # -> todos los algoritmos si se deja vacío

# Si está vacía, usar todos los algoritmos disponibles en el df
if not algorithms_to_plot:
    algorithms_to_plot = df_graphics["algorithm"].unique().tolist()

print(f"📊 Se graficarán {len(algorithms_to_plot)} algoritmos: {algorithms_to_plot}")

# === Plot por entorno ===
for env in envs:
    subdf_env = df_graphics[df_graphics["env_name"] == env]
    fig, ax = plt.subplots(figsize=(10, 6))

    # Filtrar solo los algoritmos deseados
    subdf_env = subdf_env[subdf_env["algorithm"].isin(algorithms_to_plot)]

    for algo, df_algo in subdf_env.groupby("algorithm"):
        curves = df_algo["best_curve"].tolist()
        mean, lower, upper = mean_curve_with_ci(curves, debug=False)
        if len(mean) == 0:
            continue

        x = np.arange(len(mean))
        ax.plot(x, mean, label=algo, alpha=0.9)
        ax.fill_between(x, lower, upper, alpha=0.2)

    # === Ajustes estéticos ===
    ax.set_title(f"Environment:{env}", fontweight="bold", pad=12)
    ax.set_xlabel("Generation", labelpad=10)
    ax.set_ylabel("Mean Best So Far Curve", labelpad=10)
    ax.legend(frameon=True, loc="best", fancybox=True, shadow=False, ncol=2)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(True, linestyle="--", alpha=0.5)

    # === Guardar ===
    safe_name = "".join(c if c.isalnum() or c in ('_', '-') else "_" for c in env)
    png_path = os.path.join(output_dir, f"{safe_name}.png")
    plt.savefig(png_path, bbox_inches="tight", transparent=False)
    print(f"✅ Saved: {png_path}")
    plt.close(fig)

print("\n All environment plots saved successfully!")



📊 Se graficarán 4 algoritmos: ['generic', 'macro_micro', 'mixed_generic', 'pso']
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Hyper-Dense_perturbed.png
✅ Saved: C:\Users\andre\Desktop\FTZ_Plots\Hyper-Dense_standard.png

 All environment plots saved successfully!


In [19]:
df_graphics['env_name'].unique()

array(['Hyper-Dense_perturbed', 'Hyper-Dense_standard'], dtype=object)

## 5. Hypothesis testing

In [20]:
df_wilcoxon = df_graphics[["env_name", "algorithm", "best_value"]].copy()

df_wilcoxon.head()

,env_name,algorithm,best_value
0,Hyper-Dense_perturbed,generic,17235.35
1,Hyper-Dense_perturbed,generic,17214.20
2,Hyper-Dense_perturbed,generic,17236.85
3,Hyper-Dense_perturbed,generic,17212.10
4,Hyper-Dense_perturbed,generic,17216.75


In [21]:
pairs = [
    # macro_micro vs others
    ("macro_micro", "mixed_generic"),
    ("mixed_generic", "macro_micro"),
    ("macro_micro", "generic"),
    ("generic", "macro_micro"),
    ("macro_micro", "macro"),
    ("macro", "macro_micro"),
    ("macro_micro", "micro"),
    ("micro", "macro_micro"),
    ('recomb','macro_micro'),
    ('macro_micro','recomb'),

    # mixed_generic vs others
    ("mixed_generic", "pso"),
    ("pso", "mixed_generic"),
    ("mixed_generic", "generic"),
    ("generic", "mixed_generic"),

    # pso vs others
    ("pso", "random"),
    ("random", "pso"),
]


alpha = 0.05  # significance level
results = []


In [22]:
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
import pandas as pd

alpha = 0.05
results = []

print(f"\n Performing pairwise Mann–Whitney U tests at α={alpha}\n")

# Loop over each environment
for env in df_wilcoxon["env_name"].unique():
    df_env = df_wilcoxon[df_wilcoxon["env_name"] == env]

    for a1, a2 in pairs:
        # Ensure both algorithms exist in this environment
        if a1 not in df_env["algorithm"].unique() or a2 not in df_env["algorithm"].unique():
            continue

        vals1 = df_env.loc[df_env["algorithm"] == a1, "best_value"].astype(float).values
        vals2 = df_env.loc[df_env["algorithm"] == a2, "best_value"].astype(float).values

        # Skip if too few observations
        if len(vals1) < 3 or len(vals2) < 3:
            continue

        # Mann–Whitney test (one-sided): H₁ = a1 > a2
        stat, p = mannwhitneyu(vals1, vals2, alternative="greater")

        results.append({
            "env_name": env,
            "algorithm_1": a1,
            "algorithm_2": a2,
            "p_value": p
        })

# Create DataFrame of all tests
df_mannwhitney = pd.DataFrame(results)

# Uncorrected decisions
df_mannwhitney["reject_uncorrected"] = df_mannwhitney["p_value"] < alpha

# === Apply Bonferroni correction globally across all tests ===
reject_bonf, p_adj_bonf, _, _ = multipletests(df_mannwhitney["p_value"], alpha=alpha, method="bonferroni")
df_mannwhitney["p_value_bonferroni"] = p_adj_bonf
df_mannwhitney["reject_bonferroni"] = reject_bonf

# === Summaries ===
summary = (
    df_mannwhitney.groupby(["algorithm_1", "algorithm_2"])
    .agg(
        envs_where_a1_better_uncorrected=("reject_uncorrected", "sum"),
        envs_where_a1_better_bonferroni=("reject_bonferroni", "sum")
    )
    .reset_index()
)

# === Print results ===
print("\n Detailed per-environment results:")
print(df_mannwhitney.to_string(index=False))

print("\n Summary across environments:")
print(summary.to_string(index=False))





 Performing pairwise Mann–Whitney U tests at α=0.05


 Detailed per-environment results:
             env_name   algorithm_1   algorithm_2      p_value  reject_uncorrected  p_value_bonferroni  reject_bonferroni
Hyper-Dense_perturbed   macro_micro mixed_generic 2.303619e-11                True        3.685791e-10               True
Hyper-Dense_perturbed mixed_generic   macro_micro 1.000000e+00               False        1.000000e+00              False
Hyper-Dense_perturbed   macro_micro       generic 7.094908e-15                True        1.135185e-13               True
Hyper-Dense_perturbed       generic   macro_micro 1.000000e+00               False        1.000000e+00              False
Hyper-Dense_perturbed mixed_generic           pso 7.165315e-15                True        1.146450e-13               True
Hyper-Dense_perturbed           pso mixed_generic 1.000000e+00               False        1.000000e+00              False
Hyper-Dense_perturbed mixed_generic       generic 7.1627